# 07 — Live Forecast Tracking

Logs the model's forecast for the coming week. Phase 2 will add the 'score last week's forecast vs observed' step.

Run this once per week (e.g. every Sunday). The output JSONL grows over time and feeds Phase 2's retrospective.

In [1]:
import datetime
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from body_sim import simulate
from body_sim.config import DEFAULT_PROFILE
from body_sim.validation import _seed_state

df = pd.read_parquet(REPO_ROOT / 'notebooks' / 'predictions' / 'daily_rollup.parquet')
today = datetime.date.today()
horizon = 7

In [2]:
# Use the trailing 14-day average of inputs as the 'expected' continued behaviour
recent = df.tail(14)
expected = {
    'intake_kcal': float(recent['intake_kcal'].mean()),
    'protein_g': float(recent['protein_g'].mean()),
    'carb_g': float(recent['carb_g'].mean()),
    'fat_g': float(recent['fat_g'].mean()),
    'sodium_mg': float(recent['sodium_mg'].mean()),
    'ee_hr_keytel_kcal': float(recent['ee_hr_keytel_kcal'].mean()),
    'workout_kcal': float(recent['workout_kcal'].mean()),
    'vigorous_min': int(recent['vigorous_min'].mean()),
    'intake_logged': True,
    'hr_coverage_pct': float(recent['hr_coverage_pct'].mean()),
    'steps': int(recent['steps'].mean()),
}
expected

{'intake_kcal': 1884.6923076923076,
 'protein_g': 89.48846153846154,
 'carb_g': 206.5153846153846,
 'fat_g': 73.60769230769232,
 'sodium_mg': 53.84615384615385,
 'ee_hr_keytel_kcal': 263.4295981026622,
 'workout_kcal': 70.0,
 'vigorous_min': 1,
 'intake_logged': True,
 'hr_coverage_pct': 5.054563492063493,
 'steps': 3060}

In [3]:
initial_state = _seed_state(float(df['weight_kg'].dropna().iloc[-1]))
samples = simulate.sample_parameters(n=200, seed=int(today.toordinal()))
result = simulate.simulate_forward(
    initial_state=initial_state,
    inputs_per_day=[expected.copy() for _ in range(horizon)],
    profile=DEFAULT_PROFILE,
    parameter_samples=samples,
)
band = simulate.credible_band(result.predicted_weight_kg)

forecast_records = []
for d_offset in range(horizon):
    forecast_records.append({
        'forecast_made_on': today.isoformat(),
        'target_date': (today + datetime.timedelta(days=d_offset + 1)).isoformat(),
        'predicted_weight_kg_lo': float(band['lo'][d_offset]),
        'predicted_weight_kg_median': float(band['median'][d_offset]),
        'predicted_weight_kg_hi': float(band['hi'][d_offset]),
        'inputs_used': expected,
        'phase': 1,
    })

out_path = REPO_ROOT / 'notebooks' / 'predictions' / 'live_forecasts.jsonl'
with out_path.open('a') as f:
    for rec in forecast_records:
        f.write(json.dumps(rec) + '\n')
print(f'Wrote {len(forecast_records)} forecast records to {out_path}')
print(f'Day +7 predicted weight: {band["median"][-1]:.2f} kg '
      f'(95% CI: {band["lo"][-1]:.2f}\u2013{band["hi"][-1]:.2f})')

Wrote 7 forecast records to /opt/foodlog/notebooks/predictions/live_forecasts.jsonl
Day +7 predicted weight: 83.41 kg (95% CI: 83.08–83.86)
